# 06 — Model A (A-GRU) on G0 references: agreement with the rule glosser

**Purpose.** Train text → gloss on G0 pseudo-glosses. **Be honest about what this measures:** the references are the outputs of the G0 rule, so the scores say how well the network reproduces the rule glosser — **not BdSL correctness**. Every table and figure is labelled "agreement with the rule glosser (G0 pseudo-gloss)".

**Kaggle settings.** Accelerator **GPU** · Internet **On** (pip install sacrebleu).

**Inputs.** Dataset `bdsl-k400` + **Add Input → Output of notebook 05**.

**Outputs** (under `/kaggle/working/k400/`, saved by *Save Version → Save & Run All*). `outputs/runs/G0REF_text2gloss_kaggle400_<time>/` (+ overfit checks `G0REF_text2gloss_overfit*`).

**Expected run time** [ESTIMATE, not measured yet]. 10–40 minutes on a GPU.

**What to check.** Overfit 1 and 40 reach 100% / high exact match (the code can learn); 3 seeds finished. Val WER vs G0 is mostly unseen words (they become `<unk>`).

> Honesty rules: label every number measured or estimated; results are small-scale and signer-dependent;
> the test split is used once (notebook 07); low coordinate error is not proof of correct sign language;
> G0 glosses are pseudo-glosses, not BdSL annotation; no human or expert validation exists.

In [ ]:
# Cell 1 - Setup (the same in every notebook). Finds the data and the code, sets the paths, and copies the
# outputs of earlier notebooks (added with "Add Input") into /kaggle/working/k400 so results accumulate.
import os, sys, zipfile, pathlib, json
NOTEBOOK = "06_train_model_a"

def _find_root():
    base = pathlib.Path("/kaggle/input")
    if base.is_dir():
        for dirpath, dirnames, filenames in os.walk(base):
            if "UPLOAD_MANIFEST.csv" in filenames and "PACKAGE_INFO.json" in filenames:
                return pathlib.Path(dirpath)
            depth = len(pathlib.Path(dirpath).relative_to(base).parts)
            dirnames[:] = [] if depth >= 4 else [d for d in dirnames if d not in ("videos", "k400")]
        zips = sorted(base.rglob("bdsl-k400-*.zip"))
        if zips:  # Kaggle kept the upload as one zip file: unpack it once (a few minutes)
            dest = pathlib.Path("/tmp/k400_dataset")   # /tmp is not saved as Output
            if not (dest / "UPLOAD_MANIFEST.csv").is_file():
                print("Unpacking", zips[0], "->", dest)
                zipfile.ZipFile(zips[0]).extractall(dest)
            return dest
        raise FileNotFoundError("The private dataset is not attached. Right panel: Add Input -> Your Datasets -> bdsl-k400.")
    here = pathlib.Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / "pyproject.toml").is_file() and (c / "kaggle" / "src" / "k400").is_dir():
            return c
    raise FileNotFoundError("Open this notebook inside the project folder, or on Kaggle with the dataset attached.")

DATASET_ROOT = _find_root()
CODE_ROOT = DATASET_ROOT / "code" if (DATASET_ROOT / "code" / "pyproject.toml").is_file() else DATASET_ROOT
for _p in (CODE_ROOT / "kaggle" / "src", CODE_ROOT / "src"):
    sys.path.insert(0, str(_p))
from k400.setup import setup
S = setup(DATASET_ROOT, CODE_ROOT, NOTEBOOK)

In [ ]:
S.pip_install("sacrebleu==2.6.0")
A = ["train-text2gloss", "--data", "kaggle400", "--reference", "g0"]
import glob

## Step 1 — Dry run and overfit checks (engineering checks, never results)

In [ ]:
rc_dry = S.bdsl(*A, "--max-steps", "1")
rc_o1 = S.bdsl(*A, "--overfit", "1")
rc_o40 = S.bdsl(*A, "--overfit", "40")

## Step 2 — Real training: 3 seeds, early stopping on VAL WER vs G0

In [ ]:
rc_train = S.bdsl(*A)
run_dir = pathlib.Path(sorted(d for d in glob.glob(str(S.state_root / "outputs/runs/G0REF_text2gloss_kaggle400_*")) if (pathlib.Path(d) / "metrics.json").is_file())[-1])
print("run:", run_dir.name)

## Step 3 — Curves and scores (label: agreement with the rule glosser, G0 pseudo-gloss)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
mets = json.load(open(run_dir / "metrics.json"))
meta = json.load(open(run_dir / "run.json"))
print("LABEL:", mets["label"])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for s in meta["seeds"]:
    c = pd.read_csv(run_dir / f"seed{s}/curves.csv")
    ax[0].plot(c["epoch"], c["train_loss"], label=f"seed {s}"); ax[1].plot(c["epoch"], c["monitor_wer"], label=f"seed {s}")
ax[0].set_title("training loss"); ax[1].set_title("val WER vs G0 references (%)"); [a.legend() for a in ax]; plt.tight_layout(); plt.show()
display(pd.DataFrame({sp: {k: round(v["mean"], 1) for k, v in mets["agru_mean_sd"][sp].items()} for sp in mets["agru_mean_sd"]}))

In [ ]:
S.finish(checks=[
    ("3 seeds finished", len(mets["agru"]) == 3),
    ("run is marked for the paper (clean code)", meta["for_paper"]),
    ("labelled as agreement with G0, not BdSL", "agreement with the rule glosser" in mets["label"]),
])